# Run Direct in WSL

## Check Dir

In [ ]:
import os
os.makedirs(os.path.expanduser('~/DPCC'), exist_ok=True)
print("✓ Working directory ready")

Mounted at /content/drive


## Install Miniconda into Colab (to get Python 3.10)

In [ ]:
%%bash
if [ ! -f "$HOME/miniconda.sh" ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh \
        -O ~/miniconda.sh
fi
bash ~/miniconda.sh -b -p ~/miniconda3 -u
echo "✓ Miniconda installed"
~/miniconda3/bin/conda --version

PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3
✓ Miniconda installed
conda 26.1.1


## Clone the DPCC Repo (to Drive for persistence)

In [ ]:
%%bash
cd ~/DPCC

if [ ! -d "dpcc" ]; then
    git clone --recurse-submodules "https://github.com/ghubliming/dpcc.git"
    echo "Cloned successfully."
else
    echo "Repo already exists, pulling latest..."
    cd dpcc && git pull
fi

Repo already exists, pulling latest...
Already up to date.


## Clone the D3IL

In [ ]:
# ── Check if d3il exists and is populated, clone if not ──────
D3IL="$HOME/DPCC/dpcc/d3il"
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"

echo "=== Checking d3il ==="
if [ ! -f "$D3IL/environments/d3il/setup.py" ]; then
    echo "d3il missing or empty — cloning from ALRhub..."
    rm -rf "$D3IL"
    git clone https://github.com/ALRhub/d3il.git "$D3IL"
    echo "✓ d3il cloned"
else
    echo "✓ d3il already exists, skipping clone"
fi

echo ""
echo "=== Verifying ==="
ls "$D3IL/environments/d3il/setup.py" \
    && echo "✓ d3il core setup.py found" \
    || { echo "✗ setup.py MISSING — clone may have failed"; exit 1; }

ls "$D3IL/environments/d3il/envs/gym_avoiding_env/setup.py" \
    && echo "✓ gym_avoiding_env setup.py found" \
    || { echo "✗ gym_avoiding_env setup.py MISSING"; exit 1; }

echo ""
echo "=== Installing D3IL ==="
$PIP install -e "$D3IL/environments/d3il" -q \
    && echo "✓ d3il core installed" \
    || { echo "✗ d3il core FAILED"; exit 1; }

$PIP install -e "$D3IL/environments/d3il/envs/gym_avoiding_env" -q \
    && echo "✓ gym_avoiding_env installed" \
    || { echo "✗ gym_avoiding_env FAILED"; exit 1; }

echo ""
echo "=== Done — D3IL ready ==="

## Create conda env with Python 3.10

In [ ]:
%%bash
~/miniconda3/bin/conda tos accept --override-channels \
    --channel https://repo.anaconda.com/pkgs/main
~/miniconda3/bin/conda tos accept --override-channels \
    --channel https://repo.anaconda.com/pkgs/r

~/miniconda3/bin/conda create -n dpcc python=3.10 -y -q
echo "✓ conda env 'dpcc' created with Python 3.10"
~/miniconda3/envs/dpcc/bin/python --version

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /content/miniconda3/envs/dpcc

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ld_impl_linux-64-2.44      |       h9e0c5a2_3         725 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    packaging-25.0             |  py310h06a4308_1         164 KB
    python-3.10.20             |       h741d88c_0        24.1 MB
    setuptools-80.10.2         |  py310h06a4308_0         1.3 MB
    sqlite-3.51.1              |       h3e8d24a_1         1.2 MB
    tzdata-2026a               |       h

## Install PyTorch (CUDA124)

In [ ]:
%%bash
~/miniconda3/envs/dpcc/bin/pip install \
    torch torchvision \
    --index-url https://download.pytorch.org/whl/cpu -q
echo "✓ PyTorch (CPU) installed"
~/miniconda3/envs/dpcc/bin/python -c \
    "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available())"

✓ PyTorch installed
PyTorch: 2.6.0+cu124
CUDA: True


## Install All Requirements

In [ ]:
%%bash
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"
cd ~/DPCC/dpcc

echo "========================================="
echo " INSTALLING FROM requirements.txt"
echo "========================================="
echo "Using pip: $($PIP --version)"
echo ""

if [ ! -f requirements.txt ]; then
  echo "ERROR: requirements.txt not found in $(pwd)"
  exit 1
fi

echo "Found requirements.txt -- $(wc -l < requirements.txt) packages listed"
echo ""

PASS=0
FAIL=0
SKIP=0
FAILED_PKGS=()

while IFS= read -r line || [ -n "$line" ]; do
  [[ -z "$line" || "$line" == \#* ]] && continue

  PKG="$line"
  printf "  %-40s" "$PKG"

  RESULT=$($PIP install "$PKG" -q 2>&1)
  STATUS=$?

  if [ $STATUS -eq 0 ]; then
    if echo "$RESULT" | grep -q "already satisfied"; then
      echo "[ already installed ]"
      ((SKIP++))
    else
      echo "[ installed ]"
      ((PASS++))
    fi
  else
    echo "[ FAILED ]"
    echo "    └─ $(echo "$RESULT" | tail -3)"
    FAILED_PKGS+=("$PKG")
    ((FAIL++))
  fi

done < requirements.txt

echo ""
echo "========================================="
echo " SUMMARY"
echo "========================================="
printf "  %-25s %d\n" "Newly installed:"  "$PASS"
printf "  %-25s %d\n" "Already installed:" "$SKIP"
printf "  %-25s %d\n" "Failed:"           "$FAIL"

if [ ${#FAILED_PKGS[@]} -gt 0 ]; then
  echo ""
  echo "  Failed packages:"
  for pkg in "${FAILED_PKGS[@]}"; do
    echo "    - $pkg"
  done
  echo ""
  echo "Setup incomplete. Resolve the above before proceeding."
  exit 1
else
  echo ""
  echo "All packages installed successfully."
fi

 INSTALLING FROM requirements.txt
Using pip: pip 26.0.1 from /content/miniconda3/envs/dpcc/lib/python3.10/site-packages/pip (python 3.10)

Found requirements.txt -- 144 packages listed

  absl-py==2.1.0                          [ installed ]
  accelerate==0.31.0                      [ installed ]
  cachetools==5.3.3                       [ installed ]
  casadi==3.6.5                           [ installed ]
  certifi==2024.6.2                       [ installed ]
  charset-normalizer==3.3.2               [ installed ]
  chex==0.1.86                            [ installed ]
  clarabel==0.9.0                         [ installed ]
  click==8.1.7                            [ installed ]
  cloudpickle==3.0.0                      [ installed ]
  cmeel==0.53.3                           [ installed ]
  cmeel-assimp==5.3.1                     [ installed ]
  cmeel-boost==1.83.0                     [ installed ]
  cmeel-console-bridge==1.0.2.2           [ installed ]
  cmeel-octomap==1.9.8.2      

## Install D3IL

In [ ]:
%%bash
D3IL_ROOT="$HOME/DPCC/dpcc/d3il"
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"

$PIP install -e "$D3IL_ROOT/environments/d3il" -q
$PIP install -e "$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env" -q
echo "✓ D3IL installed"

✓ D3IL installed


## Set PYTHONPATH and MuJoCo renderer

In [ ]:
import sys, os

HOME = os.path.expanduser("~")
D3IL_ROOT   = f"{HOME}/DPCC/dpcc/d3il"
GYM_AV_PATH = f"{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env"

sys.path.insert(0, D3IL_ROOT)
sys.path.insert(0, GYM_AV_PATH)

os.environ["MUJOCO_GL"]         = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
os.environ["PYTHONPATH"]        = f"{D3IL_ROOT}:{GYM_AV_PATH}:" + os.environ.get("PYTHONPATH", "")
print("✓ Paths and env vars set")

✓ Paths and env vars set


## WSL install OSMesa (If Need)

In [ ]:
# ── Step 1: Install OSMesa system library ────────────────────
sudo apt update
sudo apt install -y libosmesa6 libosmesa6-dev

# ── Step 2: Verify it's now found ────────────────────────────
echo "=== OSMesa after install ==="
ldconfig -p | grep -i osmesa

echo ""
echo "=== Python finds it now? ==="
~/miniconda3/envs/dpcc/bin/python -c "
import ctypes.util
print('OSMesa:', ctypes.util.find_library('OSMesa'))
"

# ── Step 3: Test mujoco imports cleanly ──────────────────────
echo ""
echo "=== MuJoCo test ==="
MUJOCO_GL=osmesa PYOPENGL_PLATFORM=osmesa \
    ~/miniconda3/envs/dpcc/bin/python -c "
import mujoco
print('✓ mujoco', mujoco.__version__)
"

# Here Start in Jupyter Notebook

## Full verification

In [ ]:
%%bash
PYTHON="$HOME/miniconda3/envs/dpcc/bin/python"
D3IL_ROOT="$HOME/DPCC/dpcc/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

export PYTHONPATH="$D3IL_ROOT:$GYM_AV"
export MUJOCO_GL="osmesa"
export PYOPENGL_PLATFORM="osmesa"
export LD_PRELOAD="/usr/lib/x86_64-linux-gnu/libstdc++.so.6"
export LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH"

$PYTHON - << 'EOF'
import warnings
warnings.filterwarnings("ignore")
import sys

print(f"Python:  {sys.version}")

import torch
print(f"✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

import mujoco
print(f"✓ mujoco {mujoco.__version__}")

from environments.d3il import d3il_sim
print("✓ environments.d3il.d3il_sim")

from environments.d3il.envs.gym_avoiding_env.gym_avoiding.envs.avoiding import ObstacleAvoidanceEnv
env = ObstacleAvoidanceEnv(render=False)
env.start()
obs = env.reset()
print(f"✓ ObstacleAvoidanceEnv OK | obs shape: {obs.shape}")

print("")
print("========================================")
print(" ALL CHECKS PASSED — ready to train!")
print("========================================")
EOF

Python:  3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
✓ PyTorch 2.6.0+cu124 | CUDA: True
✓ mujoco 2.3.7
✓ environments.d3il.d3il_sim
Final IK error (78 iterations):  7.2858622740627195e-06
Final IK error (0 iterations):  7.2858622740627195e-06
✓ ObstacleAvoidanceEnv OK | obs shape: (2,)

 ALL CHECKS PASSED — ready to train!



pybullet build time: Nov 28 2023 23:45:17


# Handle Gurobi

In [ ]:
import os
os.environ['GRB_LICENSE_FILE'] = os.path.expanduser('~/DPCC/gurobi.lic')

# Set Up wandb

In [ ]:
import wandb
# Either login interactively:
#wandb.login()

# Or run fully offline (no account needed):
import os
os.environ['WANDB_MODE'] = 'offline'

# Dataset download

%%bash
ZIP="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/dataset.zip"

echo "=== Top-level folders in zip ==="
unzip -l "$ZIP" | awk '{print $4}' | cut -d'/' -f1 | sort -u

echo ""
echo "=== File count per folder ==="
unzip -l "$ZIP" | awk '{print $4}' | cut -d'/' -f1 | sort | uniq -c | sort -rn

echo ""
echo "=== Total uncompressed size ==="
unzip -l "$ZIP" | tail -1

```
=== Top-level folders in zip ===

----
aligning
avoiding
inserting
Name
pushing
sorting
stacking

=== File count per folder ===
2638779 sorting
1377112 stacking
 392768 aligning
   2002 pushing
    802 inserting
     99 avoiding
      3
      1 Name
      1 ----

=== Total uncompressed size ===
17689115292                     4411562 files

Only Extract the avoiding

In [ ]:
%%bash
DATASET_DIR="$HOME/DPCC/dpcc/d3il/environments/dataset/data"
AVOIDING_DATA="$DATASET_DIR/avoiding/data"
ZIP_FILE="$DATASET_DIR/dataset.zip"

echo "========================================="
echo " D3IL DATASET SETUP"
echo " Source: ALRhub/d3il README.md"
echo " Link: https://drive.google.com/file/d/1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8"
echo "========================================="
echo ""

# ── Step 1: Check if avoiding data already extracted ─────────────
if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A $AVOIDING_DATA)" ]; then
    echo "✓ Dataset already extracted at: $AVOIDING_DATA"
    echo "  Files: $(ls $AVOIDING_DATA | wc -l) found"
    echo "  Skipping download and extraction."
    exit 0
fi

# ── Step 2: Check if zip already downloaded ───────────────────────
if [ -f "$ZIP_FILE" ]; then
    echo "✓ Zip already exists: $(du -sh $ZIP_FILE | cut -f1)"
    echo "  Skipping download."
else
    echo "Zip not found. Downloading..."
    mkdir -p "$DATASET_DIR"

    $HOME/miniconda3/envs/dpcc/bin/pip install gdown -q
    echo "✓ gdown ready"

    $HOME/miniconda3/envs/dpcc/bin/python -m gdown \
        "https://drive.google.com/uc?id=1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8" \
        -O "$ZIP_FILE"

    if [ ! -f "$ZIP_FILE" ]; then
        echo "✗ Download failed — zip file not found"
        exit 1
    fi
    echo "✓ Downloaded: $(du -sh $ZIP_FILE | cut -f1)"
fi
echo ""

# ── Step 3: Extract ONLY avoiding/ to local disk (fast) ────
echo "Extracting only avoiding/ to local disk first..."
LOCAL="/tmp/avoiding_tmp"
mkdir -p "$LOCAL"
unzip -q "$ZIP_FILE" "avoiding/*" -d "$LOCAL"

if [ $? -ne 0 ]; then
    echo "✗ Extraction failed"
    rm -rf "$LOCAL"
    exit 1
fi
echo "✓ Extracted locally: $(find $LOCAL -type f | wc -l) files | $(du -sh $LOCAL | cut -f1)"
echo ""

# ── Step 4: Copy only avoiding/ to dataset path ───────────────────
echo "Copying avoiding/ to dataset path..."
mkdir -p "$DATASET_DIR/avoiding"
cp -r "$LOCAL/avoiding/." "$DATASET_DIR/avoiding/"
echo "✓ Copied"

# ── Step 5: Cleanup local temp ────────────────────────────────────
rm -rf "$LOCAL"
echo "✓ Local temp cleaned up"
echo ""

# ── Step 6: Final verification ────────────────────────────────────
echo "========================================="
echo " VERIFICATION"
echo "========================================="
if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A $AVOIDING_DATA)" ]; then
    echo "✓ avoiding/data found"
    echo "  Files: $(ls $AVOIDING_DATA | wc -l)"
    echo "  Sample: $(ls $AVOIDING_DATA | head -3)"
    echo ""
    echo "✓ Dataset ready — you can now run the smoke test."
else
    echo "✗ avoiding/data still not found."
    echo "  Actual structure:"
    find "$DATASET_DIR" -maxdepth 3 | sort | head -20
    exit 1
fi

 D3IL DATASET SETUP
 Source: ALRhub/d3il README.md
 Link: https://drive.google.com/file/d/1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8

✓ Dataset already extracted at: /content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/avoiding/data
  Files: 96 found
  Skipping download and extraction.


## Minors before running in WSL CPU 

Uninstall triton completely — not needed for CPU

In [ ]:
# Uninstall triton completely — not needed for CPU
~/miniconda3/envs/dpcc/bin/pip uninstall triton -y

echo "✓ triton removed"

# Verify it's gone
~/miniconda3/envs/dpcc/bin/python -c "import triton" 2>&1 \
    && echo "still installed" \
    || echo "✓ triton gone"

Remove Hard Coded GPU CUDA

In [ ]:
# ── Fix 1: arrays.py — change default DEVICE to cpu ─────────
sed -i "s/DEVICE = 'cuda:0'/DEVICE = 'cpu'/" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py

sed -i "s/def batch_to_device(batch, device='cuda:0'):/def batch_to_device(batch, device='cpu'):/" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py

sed -i "s/def to_device(x, device=DEVICE):/def to_device(x, device='cpu'):/" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py

# ── Fix 2: training.py — change default train_device to cpu ──
sed -i "s/train_device='cuda'/train_device='cpu'/" \
    ~/DPCC/dpcc/diffuser/utils/training.py

sed -i "s/pin_memory=True/pin_memory=False/g" \
    ~/DPCC/dpcc/diffuser/utils/training.py

# ── Verify changes ────────────────────────────────────────────
echo "=== arrays.py ==="
grep -n "DEVICE\|batch_to_device\|to_device" \
    ~/DPCC/dpcc/diffuser/utils/arrays.py | head -10

echo ""
echo "=== training.py ==="
grep -n "train_device\|pin_memory" \
    ~/DPCC/dpcc/diffuser/utils/training.py

# Smoke Test Training Cell

In [1]:
%%bash
PYTHON="$HOME/miniconda3/envs/dpcc/bin/python"
PIP="$HOME/miniconda3/envs/dpcc/bin/pip"
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"
export MUJOCO_GL="osmesa"
export WANDB_MODE="offline"
export MPLBACKEND="agg"

cd $DPCC

# Force reinstall gymnasium and minari to resolve potential import issues
$PIP install --force-reinstall gymnasium minari -q

$PYTHON - << 'EOF'
import matplotlib
matplotlib.use('agg')   # force non-interactive backend before any other import

import torch
import diffuser.utils as utils

exp = 'avoiding-d3il'
seeds = [5]

class Parser(utils.Parser):
    dataset: str = exp
    config: str = 'config.' + exp

for seed in seeds:
    args = Parser().parse_args(experiment='diffusion', seed=seed)
    torch.manual_seed(args.seed)

    # Smoke test overrides
    args.n_train_steps     = 10
    args.n_steps_per_epoch = 10
    args.batch_size        = 2
    args.device            = 'cpu'

    print(f"Smoke test: seed={seed} | steps={args.n_train_steps} | batch={args.batch_size}")

    dataset_config = utils.Config(
        args.loader,
        savepath=(args.savepath, 'dataset_config.pkl'),
        env=args.dataset,
        horizon=args.horizon,
        normalizer=args.normalizer,
        preprocess_fns=args.preprocess_fns,
        use_padding=args.use_padding,
        max_path_length=args.max_path_length,
        include_returns=args.include_returns,
        returns_scale=args.max_path_length,
        discount=args.discount,
    )
    dataset = dataset_config()
    observation_dim = dataset.observation_dim
    action_dim      = dataset.action_dim
    goal_dim        = dataset.goal_dim

    model_config = utils.Config(
        args.model,
        savepath=(args.savepath, 'model_config.pkl'),
        horizon=args.horizon,
        transition_dim=observation_dim + action_dim,
        cond_dim=observation_dim,
        dim_mults=args.dim_mults,
        returns_condition=args.returns_condition,
        dim=args.dim,
        condition_dropout=args.condition_dropout,
        device=args.device,
    )

    diffusion_config = utils.Config(
        args.diffusion,
        savepath=(args.savepath, 'diffusion_config.pkl'),
        horizon=args.horizon,
        observation_dim=observation_dim,
        action_dim=action_dim,
        goal_dim=goal_dim,
        n_timesteps=args.n_diffusion_steps,
        loss_type=args.loss_type,
        clip_denoised=args.clip_denoised,
        predict_epsilon=args.predict_epsilon,
        action_weight=args.action_weight,
        loss_discount=args.loss_discount,
        returns_condition=args.returns_condition,
        condition_guidance_w=args.condition_guidance_w,
        device=args.device,
    )

    trainer_config = utils.Config(
        utils.Trainer,
        savepath=(args.savepath, 'trainer_config.pkl'),
        train_test_split=args.train_test_split,
        ema_decay=args.ema_decay,
        n_train_steps=args.n_train_steps,
        n_steps_per_epoch=args.n_steps_per_epoch,
        train_batch_size=args.batch_size,
        train_lr=args.learning_rate,
        gradient_accumulate_every=args.gradient_accumulate_every,
        results_folder=args.savepath,
    )

    model     = model_config()
    diffusion = diffusion_config(model)
    trainer   = trainer_config(diffusion, dataset)

    trainer.train()
    print(f"\n✓ Smoke test PASSED — seed {seed} completed {args.n_train_steps} steps")

print("\n========================================")
print(" SMOKE TEST COMPLETE — DPCC code works!")
print("========================================")
EOF

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cmeel-boost 1.83.0 requires numpy~=1.26.0; python_version >= "3.9", but you have numpy 2.2.6 which is incompatible.
qpth 0.0.17 requires numpy<2,>=1, but you have numpy 2.2.6 which is incompatible.
torch 2.10.0+cpu requires sympy>=1.13.3, but you have sympy 1.12 which is incompatible.


Smoke test: seed=5 | steps=10 | batch=2

[utils/config ] Config: <class 'diffuser.datasets.sequence.SequenceDataset'>
    discount: 0.99
    env: avoiding-d3il
    horizon: 8
    include_returns: True
    max_path_length: 150
    normalizer: LimitsNormalizer
    preprocess_fns: []
    returns_scale: 150
    use_padding: True

[ utils/config ] Saved config to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5/dataset_config.pkl

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)

[utils/config ] Config: <class 'diffuser.models.unet1d_temporal_cond.UNet1DTemporalCondModel'>
    cond_dim: 4
    condition_dropout: 0.25
    dim: 32
    dim_mults: (1, 2, 4, 8)
    horizon: 8
    returns_condition: False
    transition_dim: 6

[ utils/config ] Saved config to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5

Epoch 0: 100%|██████████| 10/10 [00:02<00:00,  3.43it/s, a0_loss=0.186, a0_loss_test=0.121, diffusion_loss=1.93, loss=0.963, loss_test=0.789, lr=1e-6, step=9]


Initial test loss:   0.7894, a0 loss:   0.1214

✓ Smoke test PASSED — seed 5 completed 10 steps

 SMOKE TEST COMPLETE — DPCC code works!


Before Real Training - change device cuda → cpu

In [ ]:
# ── Fix config file: change device cuda → cpu ────────────────
sed -i "s/'device': 'cuda'/'device': 'cpu'/" \
    ~/DPCC/dpcc/config/avoiding-d3il.py

# Also fix any 'cuda:0' variant
sed -i "s/'device': 'cuda:0'/'device': 'cpu'/" \
    ~/DPCC/dpcc/config/avoiding-d3il.py

# ── Verify ───────────────────────────────────────────────────
echo "=== config/avoiding-d3il.py device line ==="
grep -n "device" ~/DPCC/dpcc/config/avoiding-d3il.py

 # Train (using the conda env Python)

In [5]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export WANDB_MODE=offline
export MPLBACKEND=agg
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"
#                   ↑ this was missing — diffuser/ is inside here

~/miniconda3/envs/dpcc/bin/python scripts/train.py


[utils/config ] Config: <class 'diffuser.datasets.sequence.SequenceDataset'>
    discount: 0.99
    env: avoiding-d3il
    horizon: 8
    include_returns: True
    max_path_length: 150
    normalizer: LimitsNormalizer
    preprocess_fns: []
    returns_scale: 150
    use_padding: True

[ utils/config ] Saved config to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5/dataset_config.pkl

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)

[utils/config ] Config: <class 'diffuser.models.unet1d_temporal_cond.UNet1DTemporalCondModel'>
    cond_dim: 4
    condition_dropout: 0.25
    dim: 32
    dim_mults: (1, 2, 4, 8)
    horizon: 8
    returns_condition: False
    transition_dim: 6

[ utils/config ] Saved config to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/5/model_config.pkl


[utils/config ] Conf

Epoch 0:  40%|████      | 405/1000 [00:40<00:58, 10.20it/s, a0_loss=0.0547, a0_loss_test=0.109, diffusion_loss=0.951, loss=0.475, loss_test=0.757, lr=4.05e-5, step=404]

Initial test loss:   0.7571, a0 loss:   0.1095


Epoch 1:  60%|█████▉    | 595/1000 [00:58<00:38, 10.53it/s, a0_loss=0.0146, a0_loss_test=0.0343, diffusion_loss=0.439, loss=0.22, loss_test=0.389, lr=0.0001, step=1594] Traceback (most recent call last):
  File "/home/liu/DPCC/dpcc/scripts/train.py", line 131, in <module>
    trainer.train()
  File "/home/liu/DPCC/dpcc/diffuser/utils/training.py", line 188, in train
    self.train_epoch(self.n_steps_per_epoch, epoch)
  File "/home/liu/DPCC/dpcc/diffuser/utils/training.py", line 126, in train_epoch
    loss.backward()
  File "/home/liu/miniconda3/envs/dpcc/lib/python3.10/site-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
  File "/home/liu/miniconda3/envs/dpcc/lib/python3.10/site-packages/torch/autograd/__init__.py", line 364, in backward
    _engine_run_backward(
  File "/home/liu/miniconda3/envs/dpcc/lib/python3.10/site-packages/torch/autograd/graph.py", line 865, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into 

TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

# Evaluate

In [ ]:
for WSL need

In [ ]:
# ── Downgrade numpy to 1.x ───────────────────────────────────
~/miniconda3/envs/dpcc/bin/pip install "numpy<2" -q

# ── Verify ───────────────────────────────────────────────────
~/miniconda3/envs/dpcc/bin/python -c "import numpy; print('numpy', numpy.__version__)"
# Should show: numpy 1.26.x

# ── Quick test pinocchio loads ────────────────────────────────
~/miniconda3/envs/dpcc/bin/python -c "import pinocchio; print('✓ pinocchio OK')"

In [ ]:
then Run

In [8]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export WANDB_MODE=offline
export MPLBACKEND=agg                      # ← force agg, overrides Jupyter inline
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

# unset the Jupyter inline backend that leaks into %%bash subshell
unset MPLBACKEND
export MPLBACKEND=agg

$HOME/miniconda3/envs/dpcc/bin/python scripts/eval.py
$HOME/miniconda3/envs/dpcc/bin/python scripts/load_results.py

pybullet build time: Nov 28 2023 23:45:17



[ utils/serialization ] Loading model from logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/0



Traceback (most recent call last):
  File "/home/liu/DPCC/dpcc/scripts/eval.py", line 59, in <module>
    diffusion_experiment = utils.load_diffusion(args.loadbase, args.dataset, args.diffusion_loadpath, str(args.seed), epoch=args.diffusion_epoch, device=args.device)
  File "/home/liu/DPCC/dpcc/diffuser/utils/serialization.py", line 54, in load_diffusion
    dataset_config = load_config(*loadpath, 'dataset_config.pkl')
  File "/home/liu/DPCC/dpcc/diffuser/utils/serialization.py", line 36, in load_config
    config = pickle.load(open(loadpath, 'rb'))
FileNotFoundError: [Errno 2] No such file or directory: 'logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/0/dataset_config.pkl'
Traceback (most recent call last):
  File "/home/liu/DPCC/dpcc/scripts/load_results.py", line 37, in <module>
    data = np.load(f'{args.savepath}/results/halfspace_{halfspace_variant}/{variant}.npz', allow_pickle=True)
  File "/home/liu/miniconda3/envs/dpcc/lib/python3.10/site-packages/numpy/lib/npyio

CalledProcessError: Command 'b'DPCC="$HOME/DPCC/dpcc"\nD3IL_ROOT="$DPCC/d3il"\nGYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"\n\ncd $DPCC\n\nexport MUJOCO_GL=osmesa\nexport PYOPENGL_PLATFORM=osmesa\nexport WANDB_MODE=offline\nexport MPLBACKEND=agg                      # \xe2\x86\x90 force agg, overrides Jupyter inline\nexport PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"\n\n# unset the Jupyter inline backend that leaks into %%bash subshell\nunset MPLBACKEND\nexport MPLBACKEND=agg\n\n$HOME/miniconda3/envs/dpcc/bin/python scripts/eval.py\n$HOME/miniconda3/envs/dpcc/bin/python scripts/load_results.py\n'' returned non-zero exit status 1.

#  Visualize

In [10]:
%%bash
DPCC="$HOME/DPCC/dpcc"
D3IL_ROOT="$DPCC/d3il"
GYM_AV="$D3IL_ROOT/environments/d3il/envs/gym_avoiding_env"

cd $DPCC

unset MPLBACKEND
export MPLBACKEND=agg
export MUJOCO_GL=osmesa
export PYOPENGL_PLATFORM=osmesa
export PYTHONPATH="$DPCC:$D3IL_ROOT:$GYM_AV"

$HOME/miniconda3/envs/dpcc/bin/python scripts/visualize_data_constraints.py

/home/liu/DPCC/dpcc/scripts/visualize_data_constraints.py:165: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Halfspace variant: top-right-hard
Number of feasible trajectories: 1/96, 1.04%
Halfspace variant: top-left-hard
Number of feasible trajectories: 0/96, 0.00%
Halfspace variant: both-hard
Number of feasible trajectories: 2/96, 2.08%


get the png

In [ ]:
find ~/DPCC/dpcc -name "*.png" -newer ~/DPCC/dpcc/scripts/visualize_data_constraints.py \
    | sort | head -20